# Home vs Away

A comparison of shot success rates at Home vs Away.  Do players perform better at home?  That is, is the shot conversion rate higher at home?  

H0: p(away) = p(home) i.e. No difference

H1: p(away) != p(home)

In [3]:
%run ./Load_NBA_Dataset.ipynb

In [ ]:
#
import polars as pl

In [48]:

def conversion_rate(data):
    """Calculate success rate for given set of records

    Args:
    -----
    test
    Returns:
    -------
    result (dict): Dictionary of conversion rate, successful shots & total shots
    """

    made_shots = data.select(pl.sum('SHOT_MADE_FLAG')).item()
    total_shots = data.select(pl.count('SHOT_MADE_FLAG')).item()
    conversion_rate = made_shots / total_shots

    return {
        "Conversion Rate": conversion_rate, 
        "Numbers": (made_shots, total_shots)
    }

def byplayer(df):
    """
    Takes player unique identifier
    """

    # Calculate home success rate
    home_data = df.filter((pl.col('isHomeTeam') == 1))

    # Calculate away success rate
    away_data = df.filter((pl.col('isHomeTeam') == 0))

    result = [conversion_rate(home_data), conversion_rate(away_data)]

    # Add row labels
    output = pl.DataFrame(result)

    return output


### Simple Ratio Comparison

Compares the total shot conversion at home vs away for all players.

In [ ]:
#

home_all = df_combined.filter(pl.col('isHomeTeam') == 1)
away_all = df_combined.filter(pl.col('isHomeTeam') == 0)

conversion_home = home_all.select(pl.sum('SHOT_MADE_FLAG')).item() / home_all.select(pl.count('SHOT_MADE_FLAG')).item()
conversion_away = away_all.select(pl.sum('SHOT_MADE_FLAG')).item() / away_all.select(pl.count('SHOT_MADE_FLAG')).item()

print(f"The successful shot conversion rate for all players: is\n Home: {conversion_home}\n Away: {conversion_away}")

The successful shot conversion rate for all players is
 Home: 0.4701105531302121
 Away: 0.4643315756400098


### By player Comparison

Home & Away shot success (conversion) rate calculated per player.  

In [ ]:
df_byplayer = df_combined.group_by('PLAYER_ID', 'isHomeTeam').agg(
    pl.len().alias('TotalShots'),
    pl.sum('SHOT_MADE_FLAG').alias('MadeShots'),
).sort('PLAYER_ID')

df_byplayer = df_byplayer.with_columns(
    (pl.col('MadeShots') / pl.col('TotalShots')).alias('ConversionRate')
)

df_byplayer.head(5)

PLAYER_ID,isHomeTeam,TotalShots,MadeShots,ConversionRate
i64,i32,u32,i64,f64
2544,1,598,318,0.531773
2544,0,672,333,0.495536
101108,0,304,124,0.407895
101108,1,279,125,0.448029
200768,1,53,20,0.377358
…,…,…,…,…
1642502,0,2,0,0.0
1642505,1,21,9,0.428571
1642505,0,9,3,0.333333


In [86]:
# Check for missing player IDs
df_byplayer.filter(pl.col('PLAYER_ID').is_null())

PLAYER_ID,isHomeTeam,TotalShots,MadeShots,ConversionRate
i64,i32,u32,i64,f64


In [94]:
home_data = df_byplayer.filter(pl.col('isHomeTeam') == 1)
away_data = df_byplayer.filter(pl.col('isHomeTeam') == 0)

renamed = {
    "TotalShots": "TotalShots_home",
    "MadeShots": "MadeShots_home",
    "ConversionRate": "ConversionRate_home"
}

df_compare = home_data.join(away_data, on='PLAYER_ID', how='full', suffix='_away').drop('isHomeTeam', 'isHomeTeam_away', 'PLAYER_ID_away').rename(renamed)

# Remove records where a player has taken only away or home shots.  We can't compare conversion rates where a player has none.
df_compare = df_compare.drop_nulls(['TotalShots_home', 'TotalShots_away'])

df_compare = df_compare.with_columns(
    (pl.col('ConversionRate_home') - pl.col('ConversionRate_away')).alias('conversion_diff'),
    pl.when(pl.col('ConversionRate_home') > pl.col('ConversionRate_away')).then(pl.lit(1)).otherwise(pl.lit(0)).alias('betterAtHome')
)

betterHome = df_compare.select(pl.sum('betterAtHome')).item()
betterAway = df_compare.select(pl.count('betterAtHome')).item() - betterHome

print(f"Home: {betterHome}\nAway: {betterAway}")

Home: 296
Away: 247


### Statistical Significance

In [ ]:
# Simple Comparison 

# H0 - p(away) = p(home) i.e. No difference
# H1 - p(away) != p(home)

In [ ]:
# By player - include only records where statistically significant difference in conversion for home vs away.